# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Access metadata (as an object)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we show all record sets and associated fields, referencing everything by their `@id` values. Use this overview to guide your extraction and analysis steps.

In [ ]:
# List record sets and fields by their @id
record_sets = dataset.record_sets
print('Record Sets found:')
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    if 'fields' in rs:
        print('Fields:')
        for fld in rs['fields']:
            print(f"  Field @id: {fld['@id']} | name: {fld.get('name', '')} | dataType: {fld.get('dataType', '')}")
    if 'columns' in rs:
        print('Columns:')
        for col in rs['columns']:
            print(f"  Column @id: {col['@id']} | name: {col.get('name', '')}")
    print('-----')

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

* Use the Record Set and Field `@id`s as listed above to perform extraction.

In [ ]:
# Extract data from each record set
# We'll use all found record sets
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df

# Example: Show columns for the first record set
example_rs_id = record_set_ids[0] if record_set_ids else None
if example_rs_id:
    print(f"Columns in {example_rs_id}: {dataframes[example_rs_id].columns.tolist()}")
    display(dataframes[example_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
We'll demonstrate EDA on the first record set, referencing fields by their `@id`.

In [ ]:
# EDA on the first record set, referencing fields by @id
import numpy as np

if example_rs_id:
    df = dataframes[example_rs_id]

    # Find a numeric field (dataType == 'Float' or 'Integer') from Croissant schema
    rs_obj = next(rs for rs in dataset.record_sets if rs['@id'] == example_rs_id)
    numeric_fields = [fld['@id'] for fld in rs_obj.get('fields', []) if fld.get('dataType') in ['schema:Float', 'schema:Integer']]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        numeric_field_name = next(fld['name'] for fld in rs_obj['fields'] if fld['@id'] == numeric_field_id)
        print(f"Using numeric field: {numeric_field_name} (ID: {numeric_field_id})")
        # Filter: numeric_field > threshold
        threshold = 10
        if numeric_field_name in df.columns:
            filtered_df = df[df[numeric_field_name].astype(float) > threshold].copy()
            print(f"Filtered records where {numeric_field_name} > {threshold}:")
            display(filtered_df.head())

            # Normalize
            filtered_df[f"{numeric_field_name}_normalized"] = (
                filtered_df[numeric_field_name].astype(float) - filtered_df[numeric_field_name].astype(float).mean()
            ) / filtered_df[numeric_field_name].astype(float).std()
            print(f"Normalized {numeric_field_name} for filtered records:")
            display(filtered_df[[numeric_field_name, f"{numeric_field_name}_normalized"]].head())

            # Try grouping by a categorical field
            cat_fields = [fld['@id'] for fld in rs_obj.get('fields', []) if fld.get('dataType') == 'schema:Text']
            group_field_id = None
            group_field_name = None
            if cat_fields:
                group_field_id = cat_fields[0]
                group_field_name = next(fld['name'] for fld in rs_obj['fields'] if fld['@id'] == group_field_id)

            if group_field_name and group_field_name in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_name)[numeric_field_name].mean().reset_index()
                print(f"Grouped data by {group_field_name}:")
                display(grouped_df.head())
    else:
        print("No numeric field found for EDA in this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll create a simple histogram and, if possible, a bar plot grouped by a text field referencing all fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_rs_id and numeric_fields:
    df = dataframes[example_rs_id]
    numeric_field_name = next(fld['name'] for fld in dataset.record_sets[0]['fields'] if fld['@id'] == numeric_fields[0])

    # Histogram
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_name].astype(float), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_name} (field @id: {numeric_fields[0]})")
    plt.xlabel(numeric_field_name)
    plt.ylabel('Count')
    plt.show()

    # Bar plot by group field if available
    cat_fields = [fld['@id'] for fld in dataset.record_sets[0].get('fields', []) if fld.get('dataType') == 'schema:Text']
    if cat_fields:
        group_field_name = next(fld['name'] for fld in dataset.record_sets[0]['fields'] if fld['@id'] == cat_fields[0])
        plt.figure(figsize=(8,5))
        sns.barplot(
            x=group_field_name,
            y=numeric_field_name,
            data=df.groupby(group_field_name)[numeric_field_name].mean().reset_index()
        )
        plt.title(f"Mean {numeric_field_name} by {group_field_name} (field @id: {cat_fields[0]})")
        plt.xlabel(group_field_name)
        plt.ylabel(f"Mean {numeric_field_name}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated:
- Loading a FAIR^2 dataset described via Croissant schema using `mlcroissant`.
- Listing all record sets and fields by their `@id` for reliable referential access.
- Loading tabular data into DataFrames and performing basic EDA.
- Filtering and normalizing numeric fields, and grouping by categorical fields referenced by `@id`.
- Simple visualizations to explore distributions and group differences.

You can extend these steps to more detailed analyses using any field or column by their schema `@id`.